In [63]:
import pandas as pd
import numpy as np

In [2]:
'''df = pd.read_excel('/home/costela/Documentos/Escorpiao_pesa/Quadro2/dados/base_preliminar.xlsx')
ministerio = pd.read_csv('/home/costela/Documentos/Escorpiao_pesa/Quadro2/dados/pesa_ministerio_site.csv', sep=';')'''

df = pd.read_excel('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Quadro2/dados/base_preliminar.xlsx')
ministerio = pd.read_csv('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Quadro2/dados/pesa_ministerio_site.csv', sep=';')

In [3]:
# Filtra somente os pesas comm SAESc
ministerio = ministerio[ministerio['ATENDIMENTOS'].str.contains('escorp', case=False, na=False)].copy()


In [4]:
# Normaliza o texto da coluna MUNICIPIO
from unidecode import unidecode   # Importa função que remove acentos e caracteres Unicode


ministerio['MUNICIPIO'] = (               # Sobrescreve a coluna 'MUNICIPIO' com o texto padronizado
  ministerio['MUNICIPIO']
  .astype(str)                  # Converte todos os valores para texto (string)
  .apply(unidecode)             # Remove acentos e caracteres especiais Unicode
  .str.upper()                  # Transforma todo o texto em CAIXA ALTA
  .str.replace(r'[\r\n]+', ' ', regex=True) # Substitui quebras de linha (\n e \r) por um espaço
  .str.strip()                  # Remove espaços no início e fim do texto
)



In [5]:
# Contagem do número de PESA por municipio 
n_pesa = ministerio['MUNICIPIO'].value_counts(dropna=False).reset_index()
n_pesa.columns = ['PESA_SITE', 'N_PESA']

In [6]:
# Merge do numero de PESA (contagem de Pesa) ao df

df = df.merge(
    n_pesa,
    left_on="MUNI_REFERENCIADO",
    right_on="PESA_SITE",
    how="left"
)
df

,Unnamed: 0,ACESSO_LOCAL,MULTIPLO_PESA,REGIAO,PESA,MUNI_REFERENCIADO,OBSERVACOES,LAT_MUNI,LON_MUNI,LAT_PESA,...,MG_ORIGINAL,LEVE_10,MG_10,LEVE_11A59,MG_11A59,LEVE_60,MG_60,TOTAL_AMPOLAS,PESA_SITE,N_PESA
0,0,0,0,ARACATUBA,PENAPOLIS,ALTO ALEGRE,TODOS,-21.582059,-50.166198,-21.416404,...,11,11,5,159,1,109,3,23,NaN,NaN
1,1,0,0,ARACATUBA,PENAPOLIS,AVANHANDAVA,TODOS,-21.460333,-49.946516,-21.416404,...,16,12,5,166,11,49,1,36,NaN,NaN
2,2,0,0,ARACATUBA,PENAPOLIS,BARBOSA,TODOS,-21.265661,-49.951816,-21.416404,...,14,40,8,309,2,91,0,38,NaN,NaN
3,3,0,0,ARACATUBA,VALPARAISO,BENTO DE ABREU,TODOS,-21.271572,-50.811723,-21.230655,...,3,8,1,57,1,14,2,21,NaN,NaN
4,4,0,0,ARACATUBA,CLEMENTINA,BILAC,TODOS,-21.403962,-50.474640,-21.556698,...,13,15,2,155,3,77,3,41,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,640,0,0,SAO JOSE DO RIO PRETO,SAO JOSE DO RIO PRETO,IPIGUA,TODOS,0.000000,0.000000,0.000000,...,22,16,5,153,10,60,2,36,NaN,NaN
641,641,0,0,SAO JOSE DO RIO PRETO,SAO JOSE DO RIO PRETO,ONDA VERDE,TODOS,0.000000,0.000000,0.000000,...,16,35,3,165,5,40,1,9,NaN,NaN
642,642,1,0,PIRACICABA,PIRACICABA,PIRACICABA,TODOS,0.000000,0.000000,0.000000,...,171,688,81,5926,46,1209,16,452,PIRACICABA,2.0
643,643,0,0,PIRACICABA,PIRACICABA,RIO DAS PEDRAS,TODOS,0.000000,0.000000,0.000000,...,14,40,4,210,0,36,1,19,NaN,NaN


In [7]:
# As duas listas que você quer atribuir
lista_pesa_site = ['FERRAZ DE VASCONCELOS','MIRANTE DO PARANAPANEMA','PARAGUACU PAULISTA','PRESIDENTE EPITACIO',
'SANTA CRUZ DO RIO PARDO','SANTA RITA DO PASSA QUATRO','SAO JOAQUIM DA BARRA','SAO JOSE DO BARREIRO',
'SAO JOSE DO RIO PARDO','SAO LUIS DO PARAITINGA']

lista_n_pesa = [1,1,1,1,1,1,1,1,1,1,1,1]

# Filtro de municípios
filtro = df['MUNI_REFERENCIADO'].isin([
    'FERRAZ DE VASCONCELOS', 'MIRANTE DO PARANAPANEMA', 'PARAGUACU PAULISTA', 
    'PRESIDENTE EPITACIO', 'SANTA CRUZ DO RIO PARDO', 'SANTA RITA DO PASSA QUATRO', 
    'SAO JOAQUIM DA BARRA', 'SAO JOSE DO BARREIRO', 'SAO JOSE DO RIO PARDO', 
    'SAO LUIS DO PARAITINGA'])

# Atribuição simultânea zipando as duas listas
df.loc[filtro, ['PESA_SITE', 'N_PESA']] = list(zip(lista_pesa_site, lista_n_pesa))


In [8]:
# Corrige numero de PESA de SAO JOSE DO RIO PRETO
df.loc[df['PESA_SITE']=='SAO JOSE DO RIO PRETO', 'N_PESA'] = 5

In [9]:
# Preenche os nomes de todos os municipios em PESA SITE. Todos que possuem N_PESA >0 são PESA
df['PESA_SITE'] = df['MUNI_REFERENCIADO']

In [10]:
# Substitui os NAs da coluna diretamente no DataFrame
df['N_PESA'] = df['N_PESA'].fillna(0)

In [11]:
df.columns

Index(['Unnamed: 0', 'ACESSO_LOCAL', 'MULTIPLO_PESA', 'REGIAO', 'PESA',
       'MUNI_REFERENCIADO', 'OBSERVACOES', 'LAT_MUNI', 'LON_MUNI', 'LAT_PESA',
       'LON_PESA', 'DISTANCIA', 'TEMPO', 'IBGE', 'POP10', 'POP11A59', 'POP60',
       'POP_GERAL', 'TOTAL_CASOS', 'TOTAL_10A', 'TOTAL_11A59', 'TOTAL_60',
       'LEVE', 'MG', 'MG_ORIGINAL', 'LEVE_10', 'MG_10', 'LEVE_11A59',
       'MG_11A59', 'LEVE_60', 'MG_60', 'TOTAL_AMPOLAS', 'PESA_SITE', 'N_PESA'],
      dtype='str')

In [12]:
# Redefine ACESSO_LOCAL a partir da variável N_PESA. Se N_PESA > 0 ; 1, senão 0
df.loc[df['N_PESA']>0,'ACESSO_LOCAL'] = 1

In [13]:
# Corrige EMBU-GUACU (esta com acesso local, mas nao tem pesa)
df.loc[df['MUNI_REFERENCIADO']=='EMBU-GUACU', 'ACESSO_LOCAL'] = 0

In [14]:
# Corrige valores de MULTIPLO_PESA. Se N_PESA > 1, então 1; 0
df['MULTIPLO_PESA'] = 0
df.loc[df['N_PESA']>1, 'MULTIPLO_PESA'] = 1

## Continuar daqui
1. Todos os PESA dever ter tempo e distância = 0; todos os não pesa devem ter Tempo e Distância diferente de 0
2. Corrigir variavel PESA (original). Se N_PESA > 0, PESA recebe PESA_SITE
2. Finalizar mantendo somente as variaveis de interesse (talvez retirar a variavel PESA_SITE e mundar o nome do MUNI_REFERENCIADO para algo mais facil)

In [ ]:
# Pesas novos recebem tempo = 0 e distancia = 0
(df.loc[(df['ACESSO_LOCAL']==1) & (df['TEMPO']>0), ['TEMPO', 'DISTANCIA']]) = 0

In [49]:
# Define 'ITAPECERICA DA SERRA' o Pesa de 'EMBU-GUACU
df.loc[df['MUNI_REFERENCIADO']=='EMBU-GUACU', 'PESA'] = 'ITAPECERICA DA SERRA'

In [51]:
# Atribui tempo e distancia entre ITAPECERICA DA SERRA E EMBU-GUACU
df.loc[(df['ACESSO_LOCAL']==0) & (df['TEMPO']==0), ['TEMPO', 'DISTANCIA']] = [[28,18]]

In [56]:
# Municipios que nao eram Pesa em 2023  na lista do Cosems passaram a ser em 2026.   
df.loc[(df["N_PESA"] > 0) & (df["PESA"] != df["PESA_SITE"]), "PESA"] = df.loc[
    (df["N_PESA"] > 0) & (df["PESA"] != df["PESA_SITE"]), "PESA_SITE"
]

In [64]:
# Substitui por NA (NaN) onde TEMPO for igual a 0
df.loc[df["TEMPO"] == 0, ["TEMPO", "DISTANCIA"]] = np.nan

In [66]:
# Seleciona somente as colunas de interesse e salva como df
df = df.drop(columns=['Unnamed: 0','PESA_SITE'])

In [68]:
df.to_excel('dados/base_preliminar2.xlsx')